# Bronze - Ingestão VRA (Voo Regular Ativo)

Objetivo: ler os 12 csvs do volume voebem.bronze.arquivos/vra/ e materializar voebem.bronze.vra

Regras da camada: 
- sem tipagem (só string), 
- sem filtro (nenhuma linha descartada), 
- colunas de auditoria (de qual arquivo veio e quando foi ingerido), 
- idempotente (rodar o script duas vezes não duplica os dados)

In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voe_bem/bronze/arquivos/vra/*.csv"
TABELA = "voe_bem.bronze.vra"

In [0]:
bruto = (
    spark.read.format('csv')
    .option('sep', ';')
    .option('header', 'true')
    .option('skipRows', 1)
    .option('quote', '"')
    .option('escape', '"')
    .option('encoding', 'UTF-8')
    .option('mode', 'PERMISSIVE')
    .load(CAMINHO)
)

print('colunas lidas do arquivo:')
for c in bruto.columns:
    print(f" {c!r}")

# Nomes de Coluna: Delta não aceita espaço

In [0]:
RENOMEAR = {
    'ICAO Empresa Aérea': 'icao_empresa_aerea',
    'Número Voo': 'numero_voo',
    'Código Autorização (DI)': 'codigo_autorizacao',
    'Código Tipo Linha': 'codigo_tipo_linha',
    'ICAO Aeródromo Origem': 'icao_aerodromo_origem',
    'ICAO Aeródromo Destino': 'icao_aerodromo_destino',
    'Partida Prevista': 'partida_prevista',
    'Partida Real': 'partida_real',
    'Chegada Prevista': 'chegada_prevista',
    'Chegada Real': 'chegada_real',
    'Situação Voo': 'situacao_voo',
    'Código Justificativa': 'codigo_justificativa'
}

faltando = [c for c in RENOMEAR if c not in bruto.columns]
assert not faltando, f"Coluna esperada não encontrada no CSV: {faltando}"

renomeado = bruto.select(
    *[F.col(f"`{origem}`").cast('string').alias(novo) for origem, novo in RENOMEAR.items()]
)

In [0]:
import re
import unicodedata

def para_snake_case(nome):
    """Converte nome de coluna para snake_case usando regex."""
    # 1. Normalizar unicode e remover acentos
    nfkd = unicodedata.normalize('NFKD', nome)
    sem_acento = ''.join([c for c in nfkd if not unicodedata.combining(c)])
    
    # 2. Lowercase
    minusculo = sem_acento.lower()
    
    # 3. Substituir tudo que não é letra/número por underscore
    snake = re.sub(r'[^a-z0-9]+', '_', minusculo)
    
    # 4. Remover underscores no início e fim
    snake = snake.strip('_')
    
    return snake

# Aplicar a todas as colunas
renomeado_regex = bruto.select(
    *[F.col(f"`{col}`").cast('string').alias(para_snake_case(col)) 
      for col in bruto.columns]
)

print('Mapeamento original → snake_case:')
for original in bruto.columns:
    print(f"  {original!r:40} → {para_snake_case(original)!r}")

# Auditoria


In [0]:
bronze = renomeado.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name")
).withColumn(
    "_ingerido_em", F.current_timestamp()
)

# Idempotente


In [0]:
(
    bronze.write.format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(TABELA)
)

print(f"{TABELA}: {spark.table(TABELA).count():,} linhas")

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {TABELA} IS
    'Bronze - VRA (Voo Regular Ativo) da ANAC, 12 meses (ago/2025 a jul/2026).
    Dado bruto: todas as colunas string, nenhuma linha descartada.
    Carga full refresh idempotente a partir de /Volumes/voe_bem/bronze/arquivos/vra/.'    
""")

In [0]:
display(
    spark.sql(f"""
        SELECT _arquivo_origem, COUNT(*) as linhas, MAX(_ingerido_em) as ingerido_em
        FROM {TABELA}
        GROUP BY _arquivo_origem
        ORDER BY _arquivo_origem
    """)
)